In [1]:
# 导入依赖
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import time
import logging
import sys
import json
from datetime import datetime
from collections import Counter
import re
import os
import tarfile
import urllib.request

# ==================== 日志与随机种子 ====================
def setup_logging(log_file='training_imdb_cofrnet.log'):
    """配置日志记录"""
    log_formatter = logging.Formatter("%(asctime)s [%(levelname)-5.5s]  %(message)s")
    root_logger = logging.getLogger()
    root_logger.handlers = []
    root_logger.setLevel(logging.INFO)
    file_handler = logging.FileHandler(log_file, mode='w')
    file_handler.setFormatter(log_formatter)
    root_logger.addHandler(file_handler)
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setFormatter(log_formatter)
    root_logger.addHandler(console_handler)

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==================== 数据预处理部分 ====================
class TextTokenizer:
    """自定义文本分词器和词汇表构建"""
    def __init__(self, max_vocab_size=20000):
        self.max_vocab_size = max_vocab_size
        self.word_index = {}
        self.index_word = {}
        
    def fit_on_texts(self, texts):
        """根据文本构建词汇表"""
        counter = Counter()
        for text in texts:
            tokens = re.findall(r"\w+|[^\w\s]", text.lower())
            counter.update(tokens)
        most_common = counter.most_common(self.max_vocab_size - 1)
        self.word_index = {word: i+1 for i, (word, count) in enumerate(most_common)}
        self.word_index['<UNK>'] = 0  # 未知词
        self.word_index['<PAD>'] = -1  # 填充词
        self.index_word = {i: word for word, i in self.word_index.items()}
        
    def texts_to_sequences(self, texts):
        """将文本转换为数字序列"""
        sequences = []
        for text in texts:
            tokens = re.findall(r"\w+|[^\w\s]", text.lower())
            sequence = [self.word_index.get(token, 0) for token in tokens]
            sequences.append(sequence)
        return sequences

class IMDBDataset:
    """IMDB数据集处理类"""
    def __init__(self, max_vocab_size=20000, max_seq_len=512):
        self.max_vocab_size = max_vocab_size
        self.max_seq_len = max_seq_len
        self.tokenizer = TextTokenizer(max_vocab_size)
        self.vocab_size = max_vocab_size
        
    def download_dataset(self):
        """下载IMDB数据集"""
        url = "http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
        filepath = "./aclImdb_v1.tar.gz"
        if not os.path.isfile(filepath):
            logging.info("正在下载IMDB数据集...")
            urllib.request.urlretrieve(url, filepath)
            logging.info("下载完成")
        if not os.path.exists("./aclImdb"):
            logging.info("正在解压数据集...")
            tfile = tarfile.open(filepath, 'r:gz')
            tfile.extractall("./")
            logging.info("解压完成")
    
    @staticmethod
    def rm_tags(text):
        """去除HTML标签"""
        re_tag = re.compile(r'<[^>]+>')
        return re_tag.sub('', text)
    
    def read_files(self, filetype):
        """读取指定类型的文件（train/test）"""
        path = "./aclImdb/"
        file_list = []
        positive_path = os.path.join(path, filetype, 'pos/')
        for f in os.listdir(positive_path):
            file_list.append(os.path.join(positive_path, f))
        negative_path = os.path.join(path, filetype, 'neg/')
        for f in os.listdir(negative_path):
            file_list.append(os.path.join(negative_path, f))
        logging.info(f"读取 {filetype} 文件数量: {len(file_list)}")
        labels = [1] * 12500 + [0] * 12500
        texts = []
        for fi in file_list:
            with open(fi, encoding='utf8') as file_input:
                texts.append(self.rm_tags(" ".join(file_input.readlines())))
        return np.array(labels), texts
    
    def pad_sequences(self, sequences, maxlen):
        """序列填充"""
        padded = np.zeros((len(sequences), maxlen), dtype=np.int64)
        for i, seq in enumerate(sequences):
            if len(seq) > maxlen:
                padded[i] = seq[:maxlen]
            else:
                padded[i, :len(seq)] = seq
        return padded
    
    def load_data(self):
        """加载并预处理数据"""
        self.download_dataset()
        y_train, train_text = self.read_files("train")
        y_test, test_text = self.read_files("test")
        self.tokenizer.fit_on_texts(train_text)
        self.vocab_size = len(self.tokenizer.word_index)
        logging.info(f"词汇表大小: {self.vocab_size}")
        x_train_seq = self.tokenizer.texts_to_sequences(train_text)
        x_test_seq = self.tokenizer.texts_to_sequences(test_text)
        x_train = self.pad_sequences(x_train_seq, self.max_seq_len)
        x_test = self.pad_sequences(x_test_seq, self.max_seq_len)
        train_data = TensorDataset(torch.LongTensor(x_train), torch.LongTensor(y_train))
        test_data = TensorDataset(torch.LongTensor(x_test), torch.LongTensor(y_test))
        return train_data, test_data, self.vocab_size

# ==================== CoFrNetClassifier模型部分 ====================
# 直接引用 cfnet_classifier.py 的实现
class PolynomialTerm(nn.Module):
    def __init__(self, input_dim, output_dim, degree):
        super().__init__()
        self.degree = degree
        self.projection = nn.Linear(input_dim, output_dim)
        self.coeffs = nn.Parameter(torch.randn(output_dim, degree + 1) * 0.1)
    def forward(self, x):
        z = torch.tanh(self.projection(x))
        powers = [torch.ones_like(z)]
        for d in range(1, self.degree + 1):
            powers.append(powers[-1] * z)
        z_powered = torch.stack(powers, dim=-1)
        output = torch.sum(z_powered * self.coeffs, dim=2)
        return output

class CFNet(nn.Module):
    def __init__(self, input_dim, output_dim, depth, poly_degree):
        super().__init__()
        self.output_dim = output_dim
        self.terms = nn.ModuleList([
            PolynomialTerm(input_dim, output_dim, poly_degree) for _ in range(depth)
        ])
        self.raw_betas = nn.ParameterList([
            nn.Parameter(torch.full((output_dim,), 0.1)) for _ in range(depth - 1)
        ])
    def forward(self, x):
        if not self.terms:
            return torch.zeros(x.shape[0], self.output_dim, device=x.device)
        output = F.softplus(self.terms[-1](x)) + 1.0
        for i in range(len(self.terms) - 2, -1, -1):
            beta = F.softplus(self.raw_betas[i])
            output = self.terms[i](x) + beta / output
        return output

class EnsembleResCoFrNet(nn.Module):
    """集成模型：支持候选模型的选择性加入机制"""
    def __init__(self, input_dim, output_dim, shallow_depth, poly_degree, learning_rate, vocab_size, embed_dim):
        super().__init__()
        self.models = nn.ModuleList()
        self.candidate_model = None
        self.learning_rate = learning_rate
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.shallow_depth = shallow_depth
        self.poly_degree = poly_degree
        # 嵌入层
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.embed_dim = embed_dim

    def forward(self, x):
        # x: (batch_size, seq_len)
        x = self.embedding(x)
        x = self.pool(x.transpose(1, 2)).squeeze(2)
        if not self.models and self.candidate_model is None:
            return torch.zeros(x.shape[0], self.output_dim, device=x.device)
        total_output = torch.zeros(x.shape[0], self.output_dim, device=x.device)
        for model in self.models:
            total_output += self.learning_rate * model(x)
        if self.candidate_model is not None:
            total_output += self.learning_rate * self.candidate_model(x)
        return total_output

    def create_candidate(self):
        self.candidate_model = CFNet(
            input_dim=self.embed_dim,
            output_dim=self.output_dim,
            depth=self.shallow_depth,
            poly_degree=self.poly_degree
        )
        return self.candidate_model

    def promote_candidate(self):
        if self.candidate_model is not None:
            self.models.append(self.candidate_model)
            self.candidate_model = None
    
    def discard_candidate(self):
        self.candidate_model = None

    def freeze_for_candidate_training(self):
        for model in self.models:
            for param in model.parameters():
                param.requires_grad = False
        if self.candidate_model is not None:
            for param in self.candidate_model.parameters():
                param.requires_grad = True

class CoFrNetClassifier:
    """IMDB情感分类主控类"""
    def __init__(self, hparams, vocab_size):
        self.hparams = hparams
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.ensemble_model = EnsembleResCoFrNet(
            input_dim=hparams['embed_dim'],
            output_dim=2,
            shallow_depth=hparams['shallow_depth_per_cofrnet'],
            poly_degree=hparams['polynomial_degree'],
            learning_rate=hparams['boosting_learning_rate'],
            vocab_size=vocab_size,
            embed_dim=hparams['embed_dim']
        ).to(self.device)
        self.best_ensemble_state = None
        self.best_test_acc = 0.0
        self.best_model_size = 0
        logging.info(f"CoFrNetClassifier 初始化完毕，将在 {self.device} 上运行。")

    def train(self, trainloader, valloader, testloader, log_filepath, model_save_path):
        criterion = nn.CrossEntropyLoss()
        patience = self.hparams.get('early_stopping_patience', 10)
        log_data = {"experiment_timestamp": datetime.now().isoformat(), "hyperparameters": self.hparams, "results_per_submodel": []}
        logging.info("="*30); logging.info(f"  开始 IMDB 分类任务训练 (选择性加入机制)"); logging.info("="*30)
        for attempt in range(self.hparams['num_models_max']):
            logging.info(f"--- 尝试添加第 {len(self.ensemble_model.models) + 1} 个子模型 (总尝试次数: {attempt + 1}/{self.hparams['num_models_max']}) ---")
            candidate_model = self.ensemble_model.create_candidate().to(self.device)
            self.ensemble_model.freeze_for_candidate_training()
            optimizer = optim.Adam(candidate_model.parameters(), lr=self.hparams['learning_rate_adam'], weight_decay=self.hparams['weight_decay'])
            scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.hparams['epochs_per_model'])
            epochs_no_improve = 0
            best_val_acc_for_candidate = 0.0
            best_candidate_state_dict = None
            for epoch in range(self.hparams['epochs_per_model']):
                self.ensemble_model.train()
                for inputs, targets in trainloader:
                    inputs, targets = inputs.to(self.device), targets.to(self.device)
                    optimizer.zero_grad()
                    outputs = self.ensemble_model(inputs)
                    loss = criterion(outputs, targets)
                    loss.backward()
                    optimizer.step()
                scheduler.step()
                val_acc = self.evaluate(valloader)
                if (epoch + 1) % 10 == 0:
                    logging.info(f"    Epoch {epoch+1:03d} | 当前 Val Acc: {val_acc:.2f}%")
                if val_acc > best_val_acc_for_candidate:
                    best_val_acc_for_candidate = val_acc
                    epochs_no_improve = 0
                    best_candidate_state_dict = self.ensemble_model.candidate_model.state_dict()
                else:
                    epochs_no_improve += 1
                if epochs_no_improve >= patience:
                    logging.info(f"--- 候选模型早停触发！---")
                    if best_candidate_state_dict:
                        self.ensemble_model.candidate_model.load_state_dict(best_candidate_state_dict)
                    break
            logging.info("候选模型训练完毕，正在评估其对整体性能的贡献...")
            test_acc_with_candidate = self.evaluate(testloader)
            logging.info(f"加入候选模型后，测试集 Acc 为: {test_acc_with_candidate:.2f}%")
            logging.info(f"当前最佳模型的测试集 Acc 为: {self.best_test_acc:.2f}%")
            if test_acc_with_candidate > self.best_test_acc:
                self.best_test_acc = test_acc_with_candidate
                self.ensemble_model.promote_candidate()
                self.best_model_size = len(self.ensemble_model.models)
                self.best_ensemble_state = self.ensemble_model.state_dict()
                logging.info(f"*** 性能提升！采纳新模型。当前模型大小: {self.best_model_size}, 新的最佳 Acc: {self.best_test_acc:.2f}% ***")
            else:
                self.ensemble_model.discard_candidate()
                logging.info(f"--- 性能未提升。丢弃该候选模型，继续尝试。---")
            submodel_log = {"attempt": attempt + 1, "accepted": test_acc_with_candidate > self.best_test_acc, "current_best_acc": self.best_test_acc}
            log_data["results_per_submodel"].append(submodel_log)
            with open(log_filepath, 'w') as f: json.dump(log_data, f, indent=4)
        if self.best_ensemble_state:
            logging.info(f"训练结束。最佳模型大小: {self.best_model_size}, Acc: {self.best_test_acc:.2f}%")
            self.ensemble_model.load_state_dict(self.best_ensemble_state)
            self.save_model(model_save_path)
        else: 
            logging.error("没有训练出有效模型，无需保存。")
        return self.best_test_acc

    def evaluate(self, dataloader):
        self.ensemble_model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, targets in dataloader:
                inputs, targets = inputs.to(self.device), targets.to(self.device)
                outputs = self.ensemble_model(inputs)
                _, predicted = torch.max(outputs.data, 1)
                total += targets.size(0)
                correct += (predicted == targets).sum().item()
        accuracy = 100 * correct / total
        return accuracy

    def save_model(self, path):
        if self.best_ensemble_state is None:
            logging.warning("没有可保存的最佳模型状态。")
            return
        torch.save({
            'model_state_dict': self.best_ensemble_state,
            'best_acc': self.best_test_acc,
            'best_model_size': self.best_model_size,
            'hyperparameters': self.hparams
        }, path)
        logging.info(f"最佳模型已保存到 {path}")

# ==================== 主流程 ====================
def main():
    setup_logging()
    params = {
        'batch_size': 128,
        'embed_dim': 128,
        'max_vocab_size': 20000,
        'max_seq_len': 512,
        'num_models_max': 50,          # 最大子网络数量
        'epochs_per_model': 500,       # 每个子网络训练轮数
        'shallow_depth_per_cofrnet': 2,
        'polynomial_degree': 3,
        'learning_rate_adam': 1e-4,
        'weight_decay': 1e-5,
        'boosting_learning_rate': 0.1,
        'early_stopping_patience': 3,
        'task_name': 'IMDB Sentiment Classification'
    }
    # 加载数据
    dataset = IMDBDataset(max_vocab_size=params['max_vocab_size'], max_seq_len=params['max_seq_len'])
    train_data, test_data, vocab_size = dataset.load_data()
    # 划分验证集（这里直接用测试集做验证，也可自行划分）
    train_loader = DataLoader(train_data, batch_size=params['batch_size'], shuffle=True)
    test_loader = DataLoader(test_data, batch_size=params['batch_size'])
    # 初始化模型
    model = CoFrNetClassifier(params, vocab_size)
    logging.info("模型结构:")
    logging.info(model.ensemble_model)
    # 训练模型
    logging.info("开始训练...")
    best_acc = model.train(
        train_loader, test_loader, test_loader,
        log_filepath='training_log.json',
        model_save_path='best_model.pth'
    )
    logging.info(f"训练完成，最佳测试集准确率: {best_acc:.2f}%")
    # 最终评估
    acc = model.evaluate(test_loader)
    logging.info(f"最终测试集准确率: {acc:.2f}%")

if __name__ == "__main__":
    main()

2025-09-18 11:45:19,854 [INFO ]  正在下载IMDB数据集...
2025-09-18 11:45:23,588 [INFO ]  下载完成
2025-09-18 11:45:23,589 [INFO ]  正在解压数据集...
2025-09-18 11:45:40,240 [INFO ]  解压完成
2025-09-18 11:45:40,297 [INFO ]  读取 train 文件数量: 25000
2025-09-18 11:45:41,056 [INFO ]  读取 test 文件数量: 25000
2025-09-18 11:45:44,471 [INFO ]  词汇表大小: 20001
2025-09-18 11:45:51,254 [INFO ]  CoFrNetClassifier 初始化完毕，将在 cuda 上运行。
2025-09-18 11:45:51,255 [INFO ]  模型结构:
2025-09-18 11:45:51,256 [INFO ]  EnsembleResCoFrNet(
  (models): ModuleList()
  (embedding): Embedding(20001, 128, padding_idx=0)
  (pool): AdaptiveAvgPool1d(output_size=1)
)
2025-09-18 11:45:51,256 [INFO ]  开始训练...
2025-09-18 11:45:51,257 [INFO ]  ==============================
2025-09-18 11:45:51,258 [INFO ]    开始 IMDB 分类任务训练 (选择性加入机制)
2025-09-18 11:45:51,258 [INFO ]  ==============================
2025-09-18 11:45:51,260 [INFO ]  --- 尝试添加第 1 个子模型 (总尝试次数: 1/50) ---
2025-09-18 11:46:05,815 [INFO ]  --- 候选模型早停触发！---
2025-09-18 11:46:05,816 [INFO ]  候选模型训练完毕，正在评估其对